# Lesson 10 - AI Agents in Production

In this lesson you will learn **production patterns** for AI agents using the Microsoft Agent Framework (Python). We cover:

- **Παρατηρησιμότητα** — προσθήκη χρονισμού και καταγραφής στις αλληλεπιδράσεις με τον πράκτορα
- **Αξιολόγηση** — χρήση ενός πράκτορα αξιολογητή για την βαθμολόγηση της ποιότητας των απαντήσεων
- **Διαχείριση κόστους** — στρατηγικές για βελτιστοποίηση των tokens και επιλογή μοντέλου

Το σενάριο είναι ένας **ταξιδιωτικός πράκτορας** που βοηθά τους χρήστες να σχεδιάσουν ταξίδια, με παρακολούθηση και αξιολόγηση από πάνω.


## Ρύθμιση


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Παρατηρήσεις Παραγωγής

Η μετάβαση των πρακτόρων AI από πρωτότυπα σε παραγωγή απαιτεί προσεκτική προσοχή σε τρεις πυλώνες:

1. **Παρατηρησιμότητα** — Χρειάζεται να έχετε ορατότητα στο τι κάνει ο πράκτορας, πόσο χρόνο παίρνει και ποια εργαλεία καλεί. Χωρίς εντοπισμό και καταγραφή, ο εντοπισμός σφαλμάτων σε προβλήματα παραγωγής είναι σχεδόν αδύνατος.

2. **Αξιολόγηση** — Οι αυτοματοποιημένοι ποιοτικοί έλεγχοι εξασφαλίζουν ότι οι απαντήσεις του πράκτορα παραμένουν ακριβείς, πλήρεις και χρήσιμες με την πάροδο του χρόνου. Ένας πράκτορας αξιολογητής μπορεί να βαθμολογήσει τις απαντήσεις με βάση ορισμένα κριτήρια.

3. **Διαχείριση Κόστους** — Η χρήση token επηρεάζει άμεσα το κόστος. Στρατηγικές όπως η βελτιστοποίηση του prompt, η επιλογή μοντέλου και η προσωρινή αποθήκευση βοηθούν να διατηρούνται τα έξοδα υπό έλεγχο χωρίς να θυσιάζεται η ποιότητα.


## Δημιουργία ενός Παρατηρήσιμου Πράκτορα

Ορίζουμε τα εργαλεία ταξιδιού και περικλείουμε την κλήση του πράκτορα με χρονομέτρηση ώστε να μπορούμε να παρακολουθούμε την καθυστέρηση. Στην παραγωγή θα ενσωματώνατε το OpenTelemetry ή ένα παρόμοιο σύστημα ιχνηλάτησης.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Πρότυπα Αξιολόγησης

Ένα συνηθισμένο πρότυπο παραγωγής είναι η χρήση ενός δεύτερου πράκτορα ως **αξιολογητή**. Ο αξιολογητής βαθμολογεί την απάντηση του πρωτεύοντος πράκτορα με βάση προκαθορισμένα κριτήρια όπως πληρότητα, ακρίβεια και χρησιμότητα.

Αυτό επιτρέπει:
- Αυτόματες πύλες ποιότητας πριν οι απαντήσεις φτάσουν στους χρήστες
- Ανίχνευση παλινδρόμησης όταν αλλάζουν προκαλέσματα ή μοντέλα
- Συνεχή παρακολούθηση της απόδοσης του πράκτορα με την πάροδο του χρόνου


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Στρατηγικές Διαχείρισης Κόστους

Ο έλεγχος του κόστους είναι κρίσιμος για παραγωγικούς πράκτορες AI. Ακολουθούν βασικές στρατηγικές:

| Στρατηγική | Περιγραφή |
|---|---|
| **Βελτιστοποίηση εντολών** | Κρατήστε τις οδηγίες συστήματος συνοπτικές. Αφαιρέστε πλεονάζον πλαίσιο για τη μείωση των εισαγόμενων διακριτικών. |
| **Επιλογή μοντέλου** | Χρησιμοποιήστε μικρότερα, φθηνότερα μοντέλα (π.χ. GPT-4o-mini) για απλές εργασίες όπως ταξινόμηση ή εξαγωγή, και κρατήστε τα μεγαλύτερα μοντέλα για σύνθετη λογική. |
| **Αποθήκευση στην κρυφή μνήμη** | Αποθηκεύστε αποτελέσματα εργαλείων και συχνές ερωτήσεις για να αποφύγετε περιττές κλήσεις API. |
| **Προϋπολογισμοί διακριτικών** | Ορίστε όρια `max_tokens` για να αποτρέψετε απρόβλεπτα μακρές απαντήσεις. |
| **Ομαδοποίηση** | Ομαδοποιήστε πολλαπλές ερωτήσεις χρηστών σε μία κλήση API όπου είναι δυνατόν. |

Στην πράξη, μια βαθμιδωτή προσέγγιση λειτουργεί καλά: κατευθύνετε τις απλές αιτήσεις σε γρήγορο, οικονομικό μοντέλο και αναβαθμίστε μόνο τις σύνθετες ερωτήσεις σε πιο ικανό μοντέλο.


## Σύνοψη

Σε αυτό το μάθημα μάθατε πώς να:

1. **Προσθέσετε παρατηρησιμότητα** στις αλληλεπιδράσεις του πράκτορα με χρονισμό και καταγραφή, θέτοντας τα θεμέλια για την παρακολούθηση και την ιχνηλάτηση.
2. **Αξιολογείτε αυτόματα τις απαντήσεις του πράκτορα** χρησιμοποιώντας έναν πράκτορα αξιολογητή που βαθμολογεί την πληρότητα, την ακρίβεια και τη χρησιμότητα.
3. **Διαχειριστείτε τα κόστη** μέσω βελτιστοποίησης των εντολών, επιλογής μοντέλου, προσωρινής αποθήκευσης και προϋπολογισμών σε tokens.

Αυτά τα πρότυπα παραγωγής βοηθούν να διασφαλίσετε ότι οι πράκτορες AI σας είναι αξιόπιστοι, μετρήσιμοι και οικονομικά αποδοτικοί σε μεγάλη κλίμακα.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
